In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.2 Tokens

Before a model sees text it sees **numbers**. PocketLM uses the simplest
possible scheme: one integer per character.

In [ ]:
from pocketlm import CharTokenizer

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
print("vocab size:", tok.vocab_size)
ids = tok.encode("To be")
print("encode('To be') ->", ids)
print("decode back   ->", repr(tok.decode(ids)))

## Three pathologies of character tokenization

In [ ]:
# 1) Out-of-vocab characters map to <unk>, so the round trip is lossy.
oov = "\u2603"  # a snowman, not in Shakespeare
print("OOV round trip:", repr(tok.decode(tok.encode(oov))))
# 2) An emoji is its own codepoint here, also out of vocab.
print("emoji ids:", tok.encode("\U0001f600"))
# 3) Char level means long sequences: one token per character.
print("'language' needs", len(tok.encode("language")), "tokens")

This is why real models use subword tokenization (BPE): fewer tokens and
no out-of-vocab holes. We will build up to that. For in-vocab text the
char round trip is exact:

In [ ]:
assert tok.decode(tok.encode("To be")) == "To be"